In [1]:
# Cell 1: Environment setup
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [2]:
# Cell 2: Imports
import os
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import torchaudio
import random
from torch.utils.data import Dataset, DataLoader

In [3]:
# Cell 3: Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.12.0.dev20260312+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


In [ ]:
# Cell 4: Load ASVspoof data and WhatsApp voice notes
import os

# Paths
ASVSPOOF_ROOT = os.environ.get("ASVSPOOF_ROOT", r"C:\deepfake-project\data\asvspoof")
PROTOCOL_DIR  = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_cm_protocols")
TRAIN_AUDIO   = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_train", "flac")
DEV_AUDIO     = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_dev", "flac")

# Path to collected WhatsApp voice notes
WHATSAPP_VOICE_DIR = r"C:\whatsapp-voice-bot\ChrisKelleher1947.github.io\bot\collected_voice_notes"

# Verify paths exist
for path_name, path in [("ASVSPOOF_ROOT", ASVSPOOF_ROOT), ("PROTOCOL_DIR", PROTOCOL_DIR), 
                         ("TRAIN_AUDIO", TRAIN_AUDIO), ("DEV_AUDIO", DEV_AUDIO)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path_name} does not exist: {path}")

if not os.path.exists(WHATSAPP_VOICE_DIR):
    print(f"WhatsApp voice directory not found: {WHATSAPP_VOICE_DIR}")
    print("Proceeding with ASVspoof data only")
 
# Read train and dev protocol files
train_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.train.trn.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)
 
dev_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.dev.trl.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)
 
# Convert labels to integers — 0 = bonafide, 1 = spoof
train_df["label"] = train_df["label"].map({"bonafide": 0, "spoof": 1})
dev_df["label"]   = dev_df["label"].map({"bonafide": 0, "spoof": 1})
 
# Add the full file path for each audio file
train_df["path"] = train_df["file_id"].apply(lambda x: os.path.join(TRAIN_AUDIO, f"{x}.flac"))
dev_df["path"]   = dev_df["file_id"].apply(lambda x: os.path.join(DEV_AUDIO,   f"{x}.flac"))

# Add WhatsApp voice notes if directory exists
if os.path.exists(WHATSAPP_VOICE_DIR):
    whatsapp_files = [f for f in os.listdir(WHATSAPP_VOICE_DIR) if f.endswith('.ogg')]
    
    whatsapp_df = pd.DataFrame({
        "speaker": ["chris"] * len(whatsapp_files),
        "file_id": [os.path.splitext(f)[0] for f in whatsapp_files],
        "env": ["whatsapp"] * len(whatsapp_files),
        "attack": ["-"] * len(whatsapp_files),
        "label": [0] * len(whatsapp_files),
        "path": [os.path.join(WHATSAPP_VOICE_DIR, f) for f in whatsapp_files]
    })
    
    # Merge with training data
    train_df_original_count = len(train_df)
    train_df = pd.concat([train_df, whatsapp_df], ignore_index=True)
    
    print(f"  Added {len(whatsapp_df)} WhatsApp voice notes to training set")
    print(f"  ASVspoof samples: {train_df_original_count}")
    print(f"  WhatsApp samples: {len(whatsapp_df)}")
    print(f"  Total:            {len(train_df)}")
else:
    print("No WhatsApp voice notes added")

print(f"\nFinal dataset:")
print(f"Training samples:   {len(train_df)}")
print(f"Dev samples:        {len(dev_df)}")
print(f"Train label split:  {train_df['label'].value_counts().to_dict()}")
print(f"Dev label split:    {dev_df['label'].value_counts().to_dict()}")

  Added 54 WhatsApp voice notes to training set
  ASVspoof samples: 25380
  WhatsApp samples: 54
  Total:            25434

Final dataset:
Training samples:   25434
Dev samples:        24844
Train label split:  {1: 22800, 0: 2634}
Dev label split:    {1: 22296, 0: 2548}


In [ ]:
# Cell 4.5: Transform Mel-Spectogram
import torchaudio

SAMPLE_RATE = 16000

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_mels=80,
    n_fft=1024,
    hop_length=320
)

In [ ]:
# Cell 5: Dataset with WhatsApp augmentation and logging

import os
import tempfile
import subprocess
import hashlib
import numpy as np
import torch
import random
import soundfile as sf
import time
from torch.utils.data import Dataset
from scipy import signal
from collections import defaultdict

SAMPLE_RATE = 16000
MAX_SAMPLES = 64000  # 4 seconds

# Debug Settings
DEBUG = True
OPUS_DEBUG = False      
FORCE_OPUS = False
DISABLE_CACHE = True

OPUS_CACHE_DIR = None if DISABLE_CACHE else os.environ.get("OPUS_CACHE_DIR", None)

if OPUS_CACHE_DIR:
    os.makedirs(OPUS_CACHE_DIR, exist_ok=True)
    print(f"Opus compression caching enabled at: {OPUS_CACHE_DIR}")
else:
    print("Opus cache disabled")

# Global Stats Tracker
AUG_STATS = defaultdict(int)
AUG_TOTAL = 0

# Dataset class responsible for audio loading, augmentation, and feature generation
class ASVspoofDataset(Dataset):

    # Store dataset reference and augmentation settings
    def __init__(self, df, feature_extractor, augment=True):
        self.df = df
        self.feature_extractor = feature_extractor
        self.augment = augment

    # Return total dataset size
    def __len__(self):
        return len(self.df)

    # Handles loading of both wav and ogg audio files
    def load_audio(self, path):

        if path.endswith(".ogg"):
            tmp_wav = tempfile.mktemp(suffix=".wav")

            try:
                # Convert ogg audio into wav format using ffmpeg
                result = subprocess.run([
                    "ffmpeg", "-y", "-loglevel", "error",
                    "-i", path,
                    "-ar", str(SAMPLE_RATE),
                    "-ac", "1",
                    tmp_wav
                ], capture_output=True, text=True)

                if result.returncode != 0:
                    raise RuntimeError(result.stderr)

                audio, _ = sf.read(tmp_wav, dtype="float32")
                return audio

            finally:
                # Remove temporary conversion file after processing
                if os.path.exists(tmp_wav):
                    os.remove(tmp_wav)

        else:
            audio, _ = sf.read(path, dtype="float32")
            return audio

    # Applies frequency filtering
    def apply_bandpass_filter(self, audio, sr=16000, lowcut=50, highcut=400):
        nyquist = sr / 2
        low = lowcut / nyquist
        high = highcut / nyquist

        sos = signal.butter(5, [low, high], btype='band', output='sos')
        return signal.sosfilt(sos, audio).astype(np.float32)

    # Simulates WhatsApp Opus compression using ffmpeg encoding and decoding
    def apply_opus_compression(self, audio_tensor, sr=16000, source_path=None):

        # Reuse cached compressed files
        if OPUS_CACHE_DIR and source_path:
            cache_key = hashlib.md5(source_path.encode()).hexdigest()
            cache_file = os.path.join(OPUS_CACHE_DIR, f"{cache_key}.wav")

            if os.path.exists(cache_file):
                audio, _ = sf.read(cache_file, dtype="float32")
                return torch.from_numpy(audio).unsqueeze(0)

        audio_np = audio_tensor.squeeze(0).numpy()

        tmp_in = tempfile.mktemp(suffix=".wav")
        tmp_ogg = tempfile.mktemp(suffix=".ogg")
        tmp_out = tempfile.mktemp(suffix=".wav")

        try:
            sf.write(tmp_in, audio_np, sr)

            bitrate = random.choice([12, 16, 24])

            start_time = time.time()

            # Encode audio into compressed Opus format
            result = subprocess.run([
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", tmp_in,
                "-c:a", "libopus",
                "-b:a", f"{bitrate}k",
                "-ar", str(sr),
                tmp_ogg
            ], capture_output=True, text=True)

            if result.returncode != 0:
                raise RuntimeError(result.stderr)

            # Decode compressed audio back into wav format
            result = subprocess.run([
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", tmp_ogg,
                "-ar", str(sr),
                "-ac", "1",
                tmp_out
            ], capture_output=True, text=True)

            if result.returncode != 0:
                raise RuntimeError(result.stderr)

            compressed_audio, _ = sf.read(tmp_out, dtype="float32")

            elapsed = time.time() - start_time

            # Debug logging for compression and bitrate tracking
            if DEBUG and OPUS_DEBUG:
                print(f"[OPUS] {os.path.basename(source_path)} | {bitrate}kbps | {elapsed:.3f}s")

            # Cache compressed audio for future reuse
            if OPUS_CACHE_DIR and source_path:
                sf.write(cache_file, compressed_audio, sr)

            return torch.from_numpy(compressed_audio).unsqueeze(0)

        finally:
            # Remove temporary files created during compression
            for f in [tmp_in, tmp_ogg, tmp_out]:
                if f and os.path.exists(f):
                    os.remove(f)

    # Applies pitch shifting
    def apply_pitch_shift(self, audio, semitones=0):
        if semitones == 0:
            return audio

        rate = 2 ** (semitones / 12)
        indices = np.round(np.arange(0, len(audio), rate)).astype(int)
        indices = indices[indices < len(audio)]
        return audio[indices]

    # Main preprocessing and augmentation pipeline
    def __getitem__(self, idx):

        global AUG_STATS, AUG_TOTAL
        AUG_TOTAL += 1

        row = self.df.iloc[idx]

        audio = self.load_audio(row["path"]).astype(np.float32)
        audio_tensor = torch.from_numpy(audio).unsqueeze(0)

        # Skip augmentation for WhatsApp recordings
        is_whatsapp = row["path"].endswith(".ogg")

        if self.augment and not is_whatsapp:

            # Simulate compression artifacts
            if FORCE_OPUS or random.random() < 0.85:
                audio_tensor = self.apply_opus_compression(
                    audio_tensor, SAMPLE_RATE, row["path"]
                )
                AUG_STATS["opus"] += 1

            audio = audio_tensor.squeeze(0).numpy()

            # Apply random bandpass filtering
            if random.random() < 0.60:
                highcut = random.randint(350, 450)
                audio = self.apply_bandpass_filter(audio, SAMPLE_RATE, 50, highcut)
                AUG_STATS["bandpass"] += 1

            # Add random background noise
            if random.random() < 0.50:
                noise = np.random.randn(len(audio)) * random.uniform(0.003, 0.02)
                audio += noise.astype(np.float32)
                AUG_STATS["noise"] += 1

            # Apply random volume scaling
            if random.random() < 0.40:
                audio *= random.uniform(0.5, 1.5)
                AUG_STATS["volume"] += 1

            # Apply random pitch shifting
            if random.random() < 0.30:
                semitones = random.uniform(-3, 3)
                audio = self.apply_pitch_shift(audio, semitones)
                AUG_STATS["pitch"] += 1

            audio_tensor = torch.from_numpy(audio).unsqueeze(0)

        audio = audio_tensor.squeeze(0).numpy()

        # Ensure fixed input length through trimming or padding
        if len(audio) >= MAX_SAMPLES:
            audio = audio[:MAX_SAMPLES]
        else:
            audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))

        # Normalize audio amplitude before feature extraction
        peak = np.abs(audio).max()
        if peak > 0:
            audio = audio / peak

        audio = torch.from_numpy(audio).float()

        # Generate mel spectrogram features for model input
        mel = mel_transform(audio)
        mel = torch.log(mel + 1e-6)

        mel = mel.transpose(0, 1)

        max_len = MAX_SAMPLES // 320

        # Ensure consistent spectrogram dimensions across samples
        if mel.shape[0] > max_len:
            mel = mel[:max_len]
        else:
            pad_len = max_len - mel.shape[0]
            mel = torch.nn.functional.pad(mel, (0, 0, 0, pad_len))

        # Return processed spectrogram and corresponding label
        return {
            "input_values": mel.to(torch.bfloat16),
            "label": torch.tensor(row["label"], dtype=torch.long)
        }

In [ ]:
# Cell 6: Load LSTM model

import torch

# Configure model execution to use GPU
device = torch.device("cuda")

# Bidirectional LSTM classifier
class LSTMClassifier(torch.nn.Module):

    # Define recurrent feature extractor and classification layers
    def __init__(self, input_dim=80, hidden_dim=256):
        super().__init__()

        # Multi-layer bidirectional LSTM
        self.lstm = torch.nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        # Connected classification head for binary prediction
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim * 2, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(128, 2)
        )

    # Forward pass through LSTM and classifier layers
    def forward(self, x, labels=None):

        out, _ = self.lstm(x)

        # Average time features across sequence length
        out = out.mean(dim=1)

        logits = self.classifier(out)

        loss = None

        # Get classification loss during training
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(logits, labels)

        return {"loss": loss, "logits": logits}


# Initialize model and convert to bfloat16 precision
model = LSTMClassifier().to(device).to(torch.bfloat16)

# Enable TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Calculate total and trainable parameter counts
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

# Display model configuration and parameter statistics
print(f"Model loaded on {device}")
print(f"Trainable parameters: {trainable:,} / {total:,}")
print(f"Frozen parameters:    {total - trainable:,} / {total:,}")

In [ ]:
# Cell 7

from torch.utils.data import WeightedRandomSampler, DataLoader

# Create dataset objects with augmentation
train_dataset = ASVspoofDataset(train_df, feature_extractor=None, augment=True)
dev_dataset   = ASVspoofDataset(dev_df, feature_extractor=None, augment=False) # No augmentation for dev set

# Handle class imbalance using weighted sampler
class_counts = train_df["label"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = train_df["label"].map({0: class_weights[0], 1: class_weights[1]}).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    sampler=sampler,
    num_workers=0,
    pin_memory=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Training batches:   {len(train_loader)}")
print(f"Development batches: {len(dev_loader)}")
print("LSTM pipeline active (no feature_extractor required)")
print(f"Class weights: bonafide={class_weights[0]:.4f}, spoof={class_weights[1]:.4f}")

In [ ]:
# Cell 8: Optimiser and evaluation function
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from sklearn.metrics import roc_auc_score
 
# AdamW optimiser
optimiser = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01
)
 
# Linear warmup scheduler
total_steps  = len(train_loader) * 5  # 5 epochs
warmup_steps = int(0.1 * total_steps)
scheduler = LinearLR(
    optimiser,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)
 
def evaluate(model, loader, device):
    # Run model on dev set and return loss, accuracy and ROC-AUC
    model.eval()
    total_loss, correct, all_labels, all_probs = 0, 0, [], []
 
    with torch.no_grad():
        for batch in loader:
            input_values = batch["input_values"].to(device)
            labels       = batch["label"].to(device)
 
            outputs = model(input_values, labels)
            total_loss += outputs.loss.item()
 
            probs  = torch.softmax(outputs.logits, dim=-1).to(torch.float32)
            preds  = probs.argmax(dim=-1)
            correct += (preds == labels).sum().item()
 
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
 
    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset)
    roc_auc  = roc_auc_score(all_labels, all_probs)
    return avg_loss, accuracy, roc_auc
 
print("Optimiser and evaluation function ready")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps:         {warmup_steps}")

In [ ]:
from torch.amp import autocast
from sklearn.metrics import roc_auc_score

# Training configuration and checkpoint settings
EPOCHS       = 5
EVAL_STEPS   = 200
SAVE_DIR     = r"C:\deepfake-project\models\ltsm_finetuned"
best_roc_auc = 0.0
patience     = 0
PATIENCE_MAX = 4

import os

# Create model save directory if it does not already exist
os.makedirs(SAVE_DIR, exist_ok=True)

# Display training configuration summary
print("\n" + "="*60)
print("TRAINING LSTM MODEL")
print("="*60)
print(f"Save directory: {SAVE_DIR}")
print(f"Eval frequency: every {EVAL_STEPS} steps")
print(f"Early stopping patience: {PATIENCE_MAX}")
print("="*60 + "\n")


# Main training loop across all epochs
for epoch in range(EPOCHS):

    model.train()

    epoch_loss = 0
    step = 0

    AUG_STATS.clear()
    AUG_TOTAL = 0

    print(f"\n================ EPOCH {epoch+1} START ================\n")

    # Iterate through training batches
    for batch in train_loader:

        input_values = batch["input_values"].to(device)
        labels       = batch["label"].to(device)

        # Enable mixed precision training
        with autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(input_values, labels)

        loss = outputs["loss"]

        # Backpropagation and gradient update step
        loss.backward()

        # Prevent exploding gradients during LSTM training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimiser.step()
        scheduler.step()
        optimiser.zero_grad()

        epoch_loss += loss.item()
        step += 1

        # Display running training loss
        if step % 50 == 0:
            avg_loss = epoch_loss / step
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {avg_loss:.4f}")

        # Run validation evaluation at fixed intervals
        if step % EVAL_STEPS == 0:

            model.eval()

            total_loss = 0
            correct = 0
            all_labels = []
            all_probs = []

            # Disable gradient tracking during validation
            with torch.no_grad():

                for batch in dev_loader:

                    input_values = batch["input_values"].to(device)
                    labels       = batch["label"].to(device)

                    outputs = model(input_values, labels)

                    logits = outputs["logits"]
                    loss   = outputs["loss"]

                    total_loss += loss.item()

                    # Convert logits into prediction probabilities
                    probs = torch.softmax(logits, dim=-1).float()
                    preds = probs.argmax(dim=-1)

                    correct += (preds == labels).sum().item()

                    all_labels.extend(labels.detach().cpu().numpy())
                    all_probs.extend(probs[:, 1].detach().cpu().numpy())

            # Calculate validation metrics
            dev_loss = total_loss / len(dev_loader)
            dev_acc  = correct / len(dev_loader.dataset)
            dev_roc  = roc_auc_score(all_labels, all_probs)

            print(f"\n>>> Eval @ step {step}")
            print(f"Loss: {dev_loss:.4f} | Acc: {dev_acc:.4f} | ROC-AUC: {dev_roc:.4f}")

            # Save checkpoint if validation ROC-AUC improves
            if dev_roc > best_roc_auc:

                best_roc_auc = dev_roc
                patience = 0

                torch.save(
                    model.state_dict(),
                    os.path.join(SAVE_DIR, "model.pt")
                )

                print(f"New best model saved (ROC-AUC: {best_roc_auc:.4f})")

            else:
                # Increment patience counter when validation performance stalls
                patience += 1

                print(f"No improvement — patience {patience}/{PATIENCE_MAX}")

                # Stop training early if no improvement persists
                if patience >= PATIENCE_MAX:
                    print("\nEarly stopping triggered")
                    break

            model.train()

    # Display average training loss after each epoch
    print(f"\nEpoch {epoch+1} complete | Avg loss: {epoch_loss/len(train_loader):.4f}\n")

    if patience >= PATIENCE_MAX:
        break


# Display final training summary
print("\n" + "="*60)
print(f"TRAINING COMPLETE | Best ROC-AUC: {best_roc_auc:.4f}")
print("="*60)